# Практика · Межі мовних моделей> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)Цей зошит — **аудит мовної моделі**. Ми не будемо її покращувати. Ми поставимо їйчотири незручні питання й дістанемо на кожне число.1. **Чому модель не каже «не знаю».** Порахуємо в справжньому корпусі, як часто   відмова взагалі трапляється як продовження.2. **Чи можна вірити її впевненості.** Напишемо очікувану похибку калібрування   (ECE) з нуля, звіримо з бібліотечною реалізацією й побудуємо діаграму   надійності для двох моделей — обережної й жадібної.3. **Що модель запамʼятала дослівно.** Вставимо в навчальний корпус вигадані   секрети з різною кратністю повторення, навчимо мовну модель і заміряємо,   з якої кратності секрет стає видобувним.4. **Чи чесний наш поділ на частини.** Порахуємо, скільки перевірних рядків   модель уже бачила в навчанні.> ⏱ Зошит навчає дві маленькі мовні мережі. Заміряно: **128 секунд** процесорного> часу на одному потоці, без відеокарти — тобто трохи більше за дві хвилини.> Годинник покаже більше, і тим більше, чим завантаженіша машина: на нашому> прогоні при 128 секундах процесорних минуло 492 секунди справжніх. Тому час> ми міряємо `time.process_time()`, а не годинником, і останньою клітинкою> зошит друкує своє власне число.**Дані справжні** і лежать уже на твоїй машині: це українські перекладиповідомлень системних програм із `/usr/share/locale/uk/LC_MESSAGES/*.mo`. Мережане потрібна. Набір встановлених програм у кожного свій, тож **твої числавідрізнятимуться від наведених у коментарях** — форма висновків збережеться,величини ні.

## 0 · СередовищеПерше, що робить зошит із навчанням, — фіксує кількість потоків. Це не оптимізація,а умова чесного заміру часу: без фіксації потоки бібліотеки крутяться в очікуванні,і це очікування рахується як робота — процесорний час роздувається в десятки разів.Змінні оточення треба виставити **до** імпорту `numpy` і `torch`, інакше вони вжене подіють.

In [ ]:
import os
# до імпорту numpy/torch: один потік, інакше замір часу буде неправдивий
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"

import sys, re, glob, gettext, math, random, time, collections, json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sklearn
torch.set_num_threads(1)

print("Python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("torch      ", torch.__version__)
print("scikit-learn", sklearn.__version__)
print("потоків torch:", torch.get_num_threads())

NOTEBOOK_START = time.process_time()   # звідси рахуємо процесорний час усього зошита

## 1 · Дані: повідомлення програм українськоюКожна програма з графічним чи консольним інтерфейсом тримає свої переклади ускомпільованому файлі `.mo`. Модуль `gettext` зі стандартної бібліотеки вміє їхчитати: усередині лежить словник «англійський оригінал → український переклад».Беремо всі переклади, довші за 20 символів (коротші — це підписи кнопок, у нихнемає речення), і одразу ріжемо текст на слова. Токенізатор тут навмисно простий:послідовності українських літер із дозволеним апострофом усередині. Він не знаєморфології, і це нормально — усе, що ми рахуємо далі, від цього не залежить.

In [ ]:
LOCALE_DIR = "/usr/share/locale/uk/LC_MESSAGES"
WORD = re.compile(r"[а-яїієґ]+(?:[\'ʼ’][а-яїієґ]+)*")

def words(text):
    """Ріже текст на українські слова в нижньому регістрі."""
    return WORD.findall(text.lower())

def load_messages(min_len=20):
    """[(англійський оригінал, український переклад)] з усіх .mo на машині."""
    out = []
    # sorted() обовʼязковий: порядок файлів у файловій системі не гарантований,
    # а нам потрібен однаковий результат при кожному запуску
    for path in sorted(glob.glob(os.path.join(LOCALE_DIR, "*.mo"))):
        try:
            with open(path, "rb") as f:
                catalog = gettext.GNUTranslations(f)._catalog
        except Exception:
            continue                      # трапляються биті або порожні каталоги
        for src, dst in catalog.items():
            if not (isinstance(src, str) and isinstance(dst, str)):
                continue                  # службовий запис із метаданими каталогу
            if len(dst) > min_len and "Project-Id" not in dst:
                out.append((src, dst))
    return out

t0 = time.process_time()
messages = load_messages()
print(f"повідомлень із перекладом: {len(messages)}")
print(f"файлів .mo на машині:      {len(glob.glob(os.path.join(LOCALE_DIR, '*.mo')))}")
print(f"завантаження: {time.process_time() - t0:.1f} с процесорного часу")
print()
for src, dst in messages[:3]:
    print(f"  {src[:60]!r}\n  → {dst[:60]!r}\n")

## 2 · Питання перше: чому модель не каже «не знаю»Мовна модель обирає продовження за ймовірністю, а ймовірність вона оцінила зкорпусу. Отже питання «чому вона вигадує замість того, щоб відмовитись» зводитьсядо підрахунку: **як часто відмова трапляється в корпусі як продовження**.Українською відмова майже завжди починається зі слова «не», тож подивимось, щойде після нього. Порахуємо всі біграми (пари сусідніх слів), у яких перше слово —«не», і впорядкуємо продовження за частотою.

In [ ]:
unigrams = collections.Counter()
bigrams = collections.Counter()
for _src, dst in messages:
    seq = words(dst)
    unigrams.update(seq)
    for a, b in zip(seq, seq[1:]):
        bigrams[(a, b)] += 1

n_tokens = sum(unigrams.values())
print(f"слововживань у корпусі: {n_tokens}")
print(f"різних слів:            {len(unigrams)}")
print(f"біграм:                 {sum(bigrams.values())}")

In [ ]:
# усі продовження слова «не» з їхніми частотами
after_ne = collections.Counter()
for (first, second), count in bigrams.items():
    if first == "не":
        after_ne[second] += count

total_ne = sum(after_ne.values())
print(f'біграм, що починаються з «не»: {total_ne}\n')
print(f'{"продовження":<20}{"скільки":>9}{"ймовірність":>14}')
# сортуємо явно за парою (частота, слово): при однаковій частоті порядок
# має бути однаковий у кожному запуску, а не залежати від порядку вставляння
for word, count in sorted(after_ne.items(), key=lambda kv: (-kv[1], kv[0]))[:8]:
    print(f"не {word:<17}{count:>9}{count / total_ne:>14.6f}")

n_znayu = after_ne.get("знаю", 0)
print(f'\nне знаю: {n_znayu} разів, ймовірність {n_znayu / total_ne:.8f}')

Три згадки. На сотні тисяч слів. Це і є весь механізм галюцинації, зведений доодного числа: **відмова — надзвичайно рідкісний текст**, а модель відтворює частоти.Тепер порахуємо, скільки коштувало б це виправити «в лоб». Мінімум втрати мовноїмоделі досягається тоді, коли модель називає ймовірністю саме ту частку, з якоюподія трапляється в даних. Отже, домішавши в корпус *k* копій біграми «не знаю»,дістанемо ймовірність (n + k) / (усього + k), і питання «скільки треба» має точнувідповідь.

In [ ]:
best_word, best_count = max(after_ne.items(), key=lambda kv: (kv[1], kv[0]))
# скільки копій треба додати, щоб «знаю» обігнало найчастіше продовження:
# потрібно n_znayu + k > best_count
k_needed = best_count - n_znayu + 1
print(f'найчастіше продовження: «не {best_word}» — {best_count} разів')
print(f'«не знаю»:              {n_znayu} разів')
print(f'треба домішати копій «не знаю»: {k_needed}')
print(f'це {k_needed / total_ne:.4f} від усіх біграм із «не»')
if n_znayu:
    print(f'\nзараз найчастіше продовження ймовірніше за «знаю» у '
          f'{best_count / n_znayu:.1f} раза')

## 3 · Питання друге: чи можна вірити впевненостіТепер задача класифікації. Повідомлення програми буває двох родів: однірозповідають про збій, інші про успіх. Мітку візьмемо з **англійськогооригіналу** — там є слова `error`, `failed`, `cannot` і подібні, — акласифікувати будемо **український переклад**. Так мітка не є простим переписомвходу.Це не ідеальна розмітка: правило з ключових слів дірчасте (воно, наприклад, ловить`cannot`, але не `could not`). Ми беремо її саме тому, що вона справжня й дешева,і нам зараз важлива не якість класифікатора, а чесність числа поруч із йоговідповіддю.

In [ ]:
ERROR_WORDS = re.compile(
    r"\b(error|fail|failed|cannot|unable|invalid|denied|corrupt)\b", re.I)

rows = []
for src, dst in messages:
    if 4 <= len(words(dst)) <= 30:
        rows.append((dst.strip(), 1 if ERROR_WORDS.search(src) else 0))

random.Random(0).shuffle(rows)         # фіксоване зерно: поділ однаковий щоразу
n = len(rows)
cut_train, cut_hold = int(0.8 * n), int(0.9 * n)
train_rows = rows[:cut_train]
hold_rows  = rows[cut_train:cut_hold]  # відкладена: на ній добиратимемо температуру
test_rows  = rows[cut_hold:]           # перевірна: на ній лише міряємо

share_error = sum(y for _t, y in rows) / n
majority = 1 - sum(y for _t, y in test_rows) / len(test_rows)
print(f"рядків: {n} · навчальна {len(train_rows)} · відкладена {len(hold_rows)} "
      f"· перевірна {len(test_rows)}")
print(f"частка «збій»: {share_error:.4f}")
print(f"рубіж «завжди більший клас» на перевірній: {majority:.4f}")
print("\nбудь-яка точність нижча за цей рубіж не варта нічого")

### Пишемо ECE з нуля**Очікувана похибка калібрування** (ECE) рахується так:1. для кожного передбачення беремо **впевненість** — ймовірність того класу, який   модель назвала (тобто максимум із виданих ймовірностей);2. ділимо відрізок від 0 до 1 на *M* рівних кошиків і розкладаємо передбачення по   кошиках за впевненістю;3. у кожному кошику рахуємо **точність** (як часто вгадала) і **середню   впевненість** (наскільки була певна);4. беремо різницю за модулем, зважуємо на розмір кошика й додаємо.Порожні кошики пропускаємо — інакше ECE занижується тим сильніше, чим більшекошиків ми взяли.

In [ ]:
def calibration_table(probabilities, labels, n_bins=15):
    """Таблиця кошиків: (низ, верх, скільки, точність, середня впевненість).

    probabilities — масив (кількість прикладів, кількість класів).
    Порожні кошики повертаються з None замість чисел і не йдуть у суму.
    """
    confidence = probabilities.max(axis=1)          # впевненість = максимум softmax
    predicted = probabilities.argmax(axis=1)
    correct = (predicted == labels).astype(float)
    table = []
    for b in range(n_bins):
        low, high = b / n_bins, (b + 1) / n_bins
        # перший кошик замкнений з обох боків, решта — відкриті знизу,
        # щоб кожне передбачення потрапило рівно в один кошик
        if b == 0:
            inside = (confidence >= low) & (confidence <= high)
        else:
            inside = (confidence > low) & (confidence <= high)
        k = int(inside.sum())
        if k == 0:
            table.append((low, high, 0, None, None))
        else:
            table.append((low, high, k,
                          float(correct[inside].mean()),
                          float(confidence[inside].mean())))
    return table

def ece(probabilities, labels, n_bins=15):
    """Очікувана похибка калібрування: зважена сума розбіжностей по кошиках."""
    table = calibration_table(probabilities, labels, n_bins)
    total = len(labels)
    return sum(k / total * abs(accuracy - conf)
               for _lo, _hi, k, accuracy, conf in table if k)

# перевіримо на маленькому прикладі з лекції, який можна порахувати на папері
demo_conf = np.array([0.55]*4 + [0.70]*5 + [0.85]*5 + [0.95]*6)
demo_prob = np.stack([1 - demo_conf, demo_conf], axis=1)
demo_true = np.array([1]*2 + [0]*2 + [1]*3 + [0]*2 + [1]*4 + [0] + [1]*5 + [0])
print("ECE на прикладі з лекції:", round(ece(demo_prob, demo_true, n_bins=20), 4))
print("очікували:                0.0825")

### Обовʼязкова перевірка: наша реалізація проти бібліотечноїУ `scikit-learn` є `calibration_curve` — вона рахує ті самі дві величини покошиках (частку правильних і середню передбачену ймовірність), тільки дляймовірності **додатного класу**, а не для впевненості. Тож зробимо чеснепорівняння: підставимо в нашу функцію задачу, де впевненість збігається зймовірністю додатного класу, — тобто просто відберемо приклади, де модельсхиляється до класу 1.Якщо два розрахунки збігаються до чотирнадцятого знака, значить, у нашійфункції немає магії — і в бібліотечній теж.

In [ ]:
from sklearn.calibration import calibration_curve

rng = np.random.default_rng(42)
fake_prob_positive = rng.random(4000)
fake_labels = (rng.random(4000) < fake_prob_positive).astype(int)

# лише ті приклади, де модель схиляється до класу 1: там впевненість = p(клас 1)
side = fake_prob_positive > 0.5
probs2 = np.stack([1 - fake_prob_positive[side], fake_prob_positive[side]], axis=1)
ours = [row for row in calibration_table(probs2, fake_labels[side], n_bins=10) if row[2]]

lib_accuracy, lib_confidence = calibration_curve(
    fake_labels[side], fake_prob_positive[side], n_bins=10, strategy="uniform")

our_accuracy = np.array([row[3] for row in ours])
our_confidence = np.array([row[4] for row in ours])
print("наші точності по кошиках:      ", np.round(our_accuracy, 4))
print("точності sklearn:              ", np.round(lib_accuracy, 4))
assert np.allclose(our_accuracy, lib_accuracy), "точності по кошиках розійшлися!"
assert np.allclose(our_confidence, lib_confidence), "впевненості по кошиках розійшлися!"
print("\n✅ збігається")

### Дві моделі: обережна й жадібнаТепер порівняємо два класифікатори на **тих самих** ознаках — символьних n-грамахукраїнського тексту.* **обережна** — логістична регресія з типовою регуляризацією (`C=1.0`) на всіх  70 тисячах навчальних рядків. Регуляризація — це штраф за великі ваги; що  менший `C`, то сильніший штраф і то обережніші ймовірності;* **жадібна** — та сама модель майже без регуляризації (`C=1000`) на **800**  рядках. Мало даних і слабкий штраф — модель підганяє ймовірності під навчальні  приклади.Це і є механізм перевпевненості у найчистішому вигляді: модель продовжуєпокращувати ймовірності там, де вже нема чого вчитись.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                             min_df=2, max_features=50000)
X_train = vectorizer.fit_transform([t for t, _y in train_rows])
X_hold  = vectorizer.transform([t for t, _y in hold_rows])
X_test  = vectorizer.transform([t for t, _y in test_rows])
y_train = np.array([y for _t, y in train_rows])
y_hold  = np.array([y for _t, y in hold_rows])
y_test  = np.array([y for _t, y in test_rows])

t0 = time.process_time()
careful = LogisticRegression(max_iter=2000, C=1.0, random_state=0).fit(X_train, y_train)
# жадібній дістається лише 800 рядків і майже нульовий штраф за великі ваги
greedy = LogisticRegression(max_iter=5000, C=1000.0, random_state=0).fit(
    X_train[:800], y_train[:800])
print(f"навчання: {time.process_time() - t0:.1f} с процесорного часу")

models = {"обережна": careful, "жадібна": greedy}
for name, model in models.items():
    p = model.predict_proba(X_test)
    print(f"\n{name}:")
    print(f"  точність              {accuracy_score(y_test, p.argmax(1)):.4f}")
    print(f"  середня впевненість   {p.max(1).mean():.4f}")
    print(f"  ECE (15 кошиків)      {ece(p, y_test):.4f}")

Різниця між двома числами — точністю й середньою впевненістю — і є те, про щопопереджає ця тема. Подивімось на неї покошиково: **діаграма надійності** кладена одну вісь впевненість, на другу — точність усередині кошика. Каліброванамодель лягає на діагональ; стовпчик під діагоналлю означає «модель певна більше,ніж має підстав».

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (name, model) in zip(axes, models.items()):
    p = model.predict_proba(X_test)
    table = calibration_table(p, y_test, n_bins=15)
    centers = [(lo + hi) / 2 for lo, hi, k, _a, _c in table if k]
    accuracies = [a for _lo, _hi, k, a, _c in table if k]
    sizes = [k for _lo, _hi, k, _a, _c in table if k]
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="ідеальне калібрування")
    ax.bar(centers, accuracies, width=1/15*0.9, alpha=0.75,
           color="#c2185b" if name == "жадібна" else "#0f766e", label="точність у кошику")
    ax.set_title(f"{name}: ECE = {ece(p, y_test):.4f}")
    ax.set_xlabel("впевненість моделі"); ax.set_ylabel("частка правильних")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

print("скільки передбачень у кошику (жадібна модель):")
for lo, hi, k, a, c in calibration_table(greedy.predict_proba(X_test), y_test):
    if k:
        print(f"  {lo:.2f}–{hi:.2f}: {k:6} шт · точність {a:.4f} · впевненість {c:.4f}")

### Кошик «модель каже щонайменше 0.95»Найпрактичніше питання про калібрування звучить так: якщо я довірятиму лишевідповідям, де модель певна на 0.95 і більше, як часто я помилятимусь?

In [ ]:
for name, model in models.items():
    p = model.predict_proba(X_test)
    confident = p.max(1) >= 0.95
    k = int(confident.sum())
    if k:
        real = accuracy_score(y_test[confident], p.argmax(1)[confident])
        print(f"{name:10} каже ≥0.95 у {k:5} випадках із {len(y_test)} "
              f"({k/len(y_test):.2%}) · насправді права у {real:.4f}")
    else:
        print(f"{name:10} жодного разу не сказала ≥0.95")

### Температура: один скаляр, який лікує формуНайпростіший спосіб полагодити калібрування — поділити логіти на одне число *T*перед softmax. Якщо *T* > 1, розподіл згладжується й впевненість падає; якщо*T* < 1 — загострюється.Важлива умова: **температуру добирають на відкладеній частині**, а міряють наперевірній. Інакше це підглядання у відповідь. І друга: температура не міняєжодної відповіді — порядок класів від ділення на додатне число не змінюється,тож **точність лишається тією самою**. Лікується форма, а не якість.

In [ ]:
def temperature_scaled(model, X, T):
    """Ймовірності після ділення логітів на температуру."""
    logit = model.decision_function(X)               # один стовпець для двох класів
    z = np.stack([-logit / T, logit / T], axis=1)
    z = z - z.max(axis=1, keepdims=True)             # віднімаємо максимум: захист від переповнення
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(probabilities, labels):
    """Середня втрата −log p(правильний клас): саме її мінімізує підбір T."""
    p = probabilities[np.arange(len(labels)), labels]
    return float(-np.log(np.clip(p, 1e-12, None)).mean())

grid = np.exp(np.linspace(np.log(0.3), np.log(10.0), 121))
for name, model in models.items():
    scores = [(nll(temperature_scaled(model, X_hold, T), y_hold), T) for T in grid]
    best_T = min(scores)[1]
    p_before = model.predict_proba(X_test)
    p_after = temperature_scaled(model, X_test, best_T)
    print(f"{name:10} T = {best_T:.4f} · ECE {ece(p_before, y_test):.4f} → "
          f"{ece(p_after, y_test):.4f} · точність "
          f"{accuracy_score(y_test, p_before.argmax(1)):.4f} → "
          f"{accuracy_score(y_test, p_after.argmax(1)):.4f}")
print("\nT > 1 означає, що модель була перевпевнена; T < 1 — що недовпевнена")

## 4 · Питання третє: що модель запамʼятала дослівноТепер головний замір теми. Питання «що модель памʼятає про мене» не має відповіді,поки ми не знаємо, що саме лежало в корпусі. Тому зробимо інакше: **покладемо вкорпус секрет, який вигадали самі**.Такий вигаданий секрет звуть **канаркою**. Схема:1. шаблон: незмінна підказка «код доступу» плюс тіло з пʼяти випадкових слів;2. кілька канарок із різною кратністю повторення — 1, 2, 4, 16, 64, 256 копій;3. навчаємо звичайну мовну модель на корпусі з канарками;4. питаємо двома способами — чи відтворює жадібне декодування тіло дослівно, і   яке місце посідає справжнє тіло серед 500 випадкових альтернатив.Тіло беремо саме **випадковим**, а не осмисленим: осмислена фраза ймовірна самасобою, і ми заміряли б знання мови, а не памʼять.

In [ ]:
MAX_LEN = 12
PAD, BOS, EOS, UNK = 0, 1, 2, 3

# для мовної моделі беремо коротші рядки: так навчання вкладається у хвилину
lm_rows = [words(dst) for _src, dst in messages]
lm_rows = [w for w in lm_rows if 4 <= len(w) <= 10]
random.Random(0).shuffle(lm_rows)

counts = collections.Counter(w for seq in lm_rows for w in seq)
# сортуємо за парою (−частота, слово): при однаковій частоті порядок фіксований
vocabulary = ["<pad>", "<bos>", "<eos>", "<unk>"] + [
    w for w, c in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])) if c >= 10]
word_to_id = {w: i for i, w in enumerate(vocabulary)}
V = len(vocabulary)

def encode(seq):
    return [BOS] + [word_to_id.get(w, UNK) for w in seq] + [EOS]

corpus = [encode(seq) for seq in lm_rows]
print(f"рядків для мовної моделі: {len(corpus)}")
print(f"словник (слова з частотою ≥ 10): {V}")
print(f'«код» у словнику: {"код" in word_to_id} · '
      f'«доступу» у словнику: {"доступу" in word_to_id}')

In [ ]:
REPEATS = [1, 2, 4, 16, 64, 256]
PROMPT = ["код", "доступу"]
prompt_ids = [word_to_id.get(w, UNK) for w in PROMPT]
BODY_LEN = 5

canary_rng = random.Random(12345)
pool = list(range(4, V))                 # будь-які звичайні слова словника
canaries = {k: [canary_rng.choice(pool) for _ in range(BODY_LEN)] for k in REPEATS}

corpus_with_canaries = list(corpus)
for k, body in canaries.items():
    sequence = [BOS] + prompt_ids + body + [EOS]
    corpus_with_canaries += [list(sequence) for _ in range(k)]
random.Random(1).shuffle(corpus_with_canaries)

print(f"корпус: {len(corpus)} → {len(corpus_with_canaries)} рядків")
print(f"вставлено {sum(REPEATS)} копій {len(REPEATS)} різних секретів\n")
for k in REPEATS:
    body_words = " ".join(vocabulary[i] for i in canaries[k])
    print(f"  ×{k:<4} код доступу {body_words}")

In [ ]:
class TinyLM(nn.Module):
    """Той самий трансформер, що в лекціях курсу, лише маленький.

    Маска-трикутник не дає позиції зазирати вперед: модель передбачає наступне
    слово, знаючи лише попередні.
    """
    def __init__(self, vocab_size, d=64, layers=2, heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d, padding_idx=PAD)
        self.pos = nn.Embedding(MAX_LEN, d)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d,
                                           batch_first=True, dropout=0.0, norm_first=True)
        self.body = nn.TransformerEncoder(layer, layers)
        self.out = nn.Linear(d, vocab_size)

    def forward(self, x):
        length = x.size(1)
        h = self.emb(x) + self.pos(torch.arange(length))
        mask = torch.triu(torch.full((length, length), float("-inf")), 1)
        return self.out(self.body(h, mask=mask, src_key_padding_mask=(x == PAD)))

def pad_batch(batch):
    longest = max(len(x) for x in batch)
    return torch.tensor([x + [PAD] * (longest - len(x)) for x in batch])

def train_lm(data, steps, seed, lr=3e-3, batch_size=32):
    torch.manual_seed(seed)
    model = TinyLM(V)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    r = random.Random(seed)
    indices = list(range(len(data)))
    started = time.process_time()
    for _ in range(steps):
        x = pad_batch([data[j] for j in r.sample(indices, batch_size)])
        loss = loss_fn(model(x[:, :-1]).reshape(-1, V), x[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    return model, float(loss.item()), time.process_time() - started

print(f"ваг у моделі: {sum(p.numel() for p in TinyLM(V).parameters())}")

In [ ]:
@torch.no_grad()
def log_probability(model, body):
    """Логарифм імовірності тіла секрету після підказки «код доступу».

    Рахуємо покроково: на кожному кроці беремо логарифм імовірності саме того
    слова, яке має стояти далі, і додаємо. Сума логарифмів — логарифм добутку.
    """
    sequence = [BOS] + prompt_ids
    total = 0.0
    for token in body:
        x = torch.tensor([sequence[-MAX_LEN:]])
        logits = model(x)[0, len(sequence) - 1]
        total += float(torch.log_softmax(logits, -1)[token])
        sequence.append(token)
    return total

@torch.no_grad()
def greedy_continue(model, how_many=BODY_LEN):
    """Що модель скаже після «код доступу», якщо щоразу брати найімовірніше слово."""
    sequence = [BOS] + prompt_ids
    for _ in range(how_many):
        x = torch.tensor([sequence[-MAX_LEN:]])
        sequence.append(int(model(x)[0, len(sequence) - 1].argmax()))
    return sequence[len(prompt_ids) + 1:]

N_ALTERNATIVES = 500
SEEDS = 2
STEPS = 1500

results = collections.defaultdict(lambda: collections.defaultdict(list))
for seed in range(SEEDS):
    model, last_loss, seconds = train_lm(corpus_with_canaries, STEPS, seed)
    model.eval()

    alt_rng = random.Random(777 + seed)
    alternatives = [[alt_rng.choice(pool) for _ in range(BODY_LEN)]
                    for _ in range(N_ALTERNATIVES)]
    alt_scores = sorted((log_probability(model, a) for a in alternatives), reverse=True)
    said = greedy_continue(model)

    for k, body in canaries.items():
        score = log_probability(model, body)
        rank = sum(1 for s in alt_scores if s > score) + 1
        results[k]["rank"].append(rank)
        results[k]["exposure"].append(math.log2(N_ALTERNATIVES / rank))
        results[k]["verbatim"].append(int(said == body))
    print(f"зерно {seed}: втрата {last_loss:.3f} · {seconds:.0f} с процесорних · "
          f'модель каже «код доступу {" ".join(vocabulary[i] for i in said)}»')

In [ ]:
print(f'{"кратність":<12}{"ранг із 500":<16}{"викритість, біт":<18}дослівно')
for k in REPEATS:
    r = results[k]
    ranks = f'{min(r["rank"])}..{max(r["rank"])}'
    exposure = sum(r["exposure"]) / len(r["exposure"])
    hits = f'{sum(r["verbatim"])}/{SEEDS}'
    print(f"×{k:<11}{ranks:<16}{exposure:<18.2f}{hits}")

In [ ]:
# те саме, але видно кожне зерно окремо: різниця між зернами — теж результат
for k in REPEATS:
    print(f'×{k:<5} ранги по зернах {results[k]["rank"]} · '
          f'викритість {[round(e, 2) for e in results[k]["exposure"]]}')

ceiling = math.log2(N_ALTERNATIVES)
print(f"\nстеля цього заміру: log2({N_ALTERNATIVES}) = {ceiling:.2f} біта")
print("ранг 1 означає «викритість не менша за стелю», а не «рівно стільки»")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
exposures = [sum(results[k]["exposure"]) / SEEDS for k in REPEATS]
verbatim = [sum(results[k]["verbatim"]) for k in REPEATS]
colours = ["#c2185b" if v else "#0f766e" for v in verbatim]
ax.bar(range(len(REPEATS)), exposures, color=colours, alpha=0.8)
ax.axhline(ceiling, ls="--", color="grey", lw=1)
ax.text(0.02, ceiling - 0.35, f"стеля заміру {ceiling:.2f} біта", fontsize=9, color="grey")
ax.set_xticks(range(len(REPEATS)))
ax.set_xticklabels([f"×{k}" for k in REPEATS])
ax.set_xlabel("скільки разів секрет повторено в корпусі")
ax.set_ylabel("викритість, біт")
ax.set_title("рожевим — там, де жадібне декодування відтворило секрет дослівно")
plt.tight_layout(); plt.show()

## 5 · Питання четверте: чи чесний наш поділОстаннє. Ми ділили дані на частини після перемішування з фіксованим зерном —процедура бездоганна. Але перемішування не рятує від **повторів у самомукорпусі**: різні програми перекладають однакові повідомлення, і після поділу однакопія лишається в навчальній частині, а друга їде в перевірну.Порахуємо, скільки перевірних рядків модель уже бачила, і чи краща вона на них.

In [ ]:
train_texts = collections.Counter(t for t, _y in train_rows)
leaked = np.array([t in train_texts for t, _y in test_rows])

print(f"різних текстів у навчальній частині: {len(train_texts)} із {len(train_rows)}")
print(f"текстів, що трапляються не раз:      "
      f"{sum(1 for _t, c in train_texts.items() if c > 1)}")
print(f"\nперевірних рядків, які дослівно є в навчальній: "
      f"{int(leaked.sum())} із {len(test_rows)} ({leaked.mean():.4f})")

p_test = careful.predict_proba(X_test).argmax(1)
acc_all = accuracy_score(y_test, p_test)
acc_leaked = accuracy_score(y_test[leaked], p_test[leaked])
acc_clean = accuracy_score(y_test[~leaked], p_test[~leaked])
print(f"\nточність на всій перевірній: {acc_all:.4f}")
print(f"точність на протеклих:       {acc_leaked:.4f}  ({int(leaked.sum())} шт)")
print(f"точність на чистих:          {acc_clean:.4f}  ({int((~leaked).sum())} шт)")
print(f"завищення через витік:       {acc_all - acc_clean:+.4f}")

# завищення = частка протеклих × перевага на них. Перевіримо рівність числом.
predicted_inflation = leaked.mean() * (acc_leaked - acc_clean)
print(f"\nперевірка: частка × перевага = {leaked.mean():.4f} × "
      f"{acc_leaked - acc_clean:+.4f} = {predicted_inflation:+.4f}")
assert abs(predicted_inflation - (acc_all - acc_clean)) < 1e-9, "рівність не зійшлася!"
print("✅ рівність сходиться")

In [ ]:
spent = time.process_time() - NOTEBOOK_START
print(f"процесорного часу на весь зошит: {spent:.0f} с = {spent/60:.1f} хв")
print("годинник показав би більше, якщо машина зайнята іншими справами")

## 6 · Що ми дісталиЧотири числа, і жодне не було відоме до заміру.1. **Відмова майже не трапляється в корпусі** — а модель відтворює частоти. Щоб   «не знаю» стало найімовірнішим продовженням слова «не», його довелося б   домішати в корпус десятки тисяч разів. Це і є причина, чому вигадка дешевша   за відмову.2. **Впевненість і правота — різні речі, і розрив вимірний.** Дві моделі на тих   самих ознаках дали різні ECE, а один скаляр — температура — зменшив розрив,   не змінивши жодної відповіді.3. **Секрет, повторений у корпусі, стає видобувним**, і кратність повторення   вирішує. Наш замір показує, з якої саме — і чесно називає стелю: із   500 альтернативами вище за 8.97 біта заміряти нічого.4. **Наш власний поділ на частини протік**, і завищення точності від цього   рівне добутку двох чисел, які ми порахували окремо.Найважливіше з цього — звʼязок між пунктами 3 і 4. Здатність запамʼятовувати йвразливість оцінювання до витоку — це **одне явище**: модель, яка добрезапамʼятовує, і завчить секрет, і покаже завищену метрику на протеклих рядках.## 7 · ЗавданняПовний текст із критеріями «зроблено» — у [homework.html](homework.html).**🟢 Рівень 1.** Заміряй калібрування ще однієї моделі з тими самими ознаками —наприклад `LogisticRegression(C=0.01)`. Побудуй їй діаграму надійності й скажи,у який бік від діагоналі вона лягла. Підказка: сильна регуляризація зазвичай даєнедовпевненість, тобто протилежний бік.**🟡 Рівень 2.** Постав канарку на кратностях, яких тут немає (8, 32, 128), ізнайди, де саме проходить межа дослівного відтворення на твоєму корпусі. Порівняйіз межею, яку дав цей зошит.**🔴 Рівень 3.** Повтори замір витоку з **нормалізацією**: зведи текст до нижньогорегістру, стисни пробіли, прибери розділові знаки — і подивись, наскільки більшерядків виявиться протеклими. Потім перевір, чи змінилось від цього завищенняточності, і поясни напрямок зміни.